# Musicology prompt-intervention eval — Colab runner

Runs the WeaveMuse manager agent over `data/eval/tasks_musicology.jsonl` under the
`default` vs `expert` prompt variants, captures traces, and scores them with an LLM judge.
See `weavemuse/eval/README.md` and `weavemuse/eval/STUDY_LOG.md`.

**Before you start:**
1. **Runtime → Change runtime type → GPU** (T4 is fine if you pass a smaller `--model-id`; the
   30B default needs an A100/L4). If `nvidia-smi` below says *command not found*, you are on a CPU runtime.
2. The repo is **private** — the clone cell asks for a GitHub personal access token (classic PAT with
   `repo` scope, or a fine-grained token with read access to `noamsprei/LAFS-weavemuse`).
3. **Confidentiality:** the Didone CSVs are not in the repo; you place them on this VM yourself
   (Drive mount). That puts the data on Google infrastructure for the session — only proceed if
   that is acceptable for your data-handling constraints.

In [ ]:
# Expect a GPU line here. 'command not found' => switch to a GPU runtime and rerun.
!nvidia-smi || echo 'NO GPU RUNTIME — Runtime > Change runtime type > GPU'

In [ ]:
# Clone the private PR branch. Paste a GitHub PAT with read access to the repo.
import os
from getpass import getpass

_pat = getpass("GitHub PAT: ").strip()
_url = f"https://{_pat}@github.com/noamsprei/LAFS-weavemuse.git"
!git clone --branch eval-harness --single-branch "$_url" LAFS-weavemuse
del _pat, _url
%cd /content/LAFS-weavemuse
!git log --oneline -3

In [ ]:
# Install. torch is already present on Colab; this pulls smolagents, pandas,
# the remote tool clients, and the judge's anthropic/openai clients.
!pip install -q -e ".[remote]"
print("\nif pip printed a 'restart runtime' notice: Runtime > Restart, then rerun from the %cd cell")

In [ ]:
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HF token (for the remote NotaGen/StableAudio/Flamingo tools): ")
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (remote judge; blank = judge locally): ")

## Didone data

Point `DIDONE_DATA_DIR` at a folder containing:
```
metadata.csv
harmony_analysis/tonal_plan_overview.csv
harmony_analysis/tonal_plan_segments.csv
harmony_analysis/parsed_harmony_events.csv
text_tonal_alignment/section_tonal_plan.csv
textual_plan/textual_plan_overview.csv
textual_plan/textual_plan_sections.csv
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# adjust to where you put the data in your Drive
os.environ["DIDONE_DATA_DIR"] = "/content/drive/MyDrive/didone_data"
!ls -la "$DIDONE_DATA_DIR" && echo '---' && ls "$DIDONE_DATA_DIR"/harmony_analysis

In [ ]:
# Sanity check: tools load and return data for a sample aria
from weavemuse.tools.didone_tools import didone_tools

t = {x.name: x for x in didone_tools()}
print(t["get_tonal_plan"].forward("0012"))
print("\n---\n")
print(t["get_tonal_plans"].forward("0001, 0012, 0041"))

## Run the sweep

First pass = the 33 `core` tasks (SW1 modulation, SW2 cadences, MW1 period-norm) × 2 variants = 66 runs.
Drop the `--task-ids` line for the full 56-task / 112-run sweep.

In [ ]:
import json, subprocess

core = [json.loads(l)["task_id"] for l in open("data/eval/tasks_musicology.jsonl")
        if "core" in json.loads(l)["tags"]]

cmd = [
    "python", "scripts/run_eval.py",
    "--dataset", "data/eval/tasks_musicology.jsonl",
    "--variants", "data/eval/variants_musicology.json",
    "--run-id", "colab_core",
    "--tool-mode", "remote",
    "--max-steps", "12",
    "--task-ids", ",".join(core),
    # "--model-id", "Qwen/Qwen2.5-7B-Instruct",  # uncomment on a 16GB T4
]
subprocess.run(cmd, check=True)

In [ ]:
# Score the traces. remote => needs ANTHROPIC_API_KEY; local => same local model (smoke only).
backend = "remote" if os.environ.get("ANTHROPIC_API_KEY") else "local"
subprocess.run([
    "python", "scripts/run_judge.py",
    "--traces-dir", "outputs/eval/colab_core/traces",
    "--backend", backend,
    "--rubric", "weavemuse/eval/rubrics/default.json",
], check=True)

In [ ]:
# Persist results off the ephemeral VM
!cp -r outputs/eval/colab_core "/content/drive/MyDrive/musicology_eval_colab_core"
!python -m json.tool outputs/eval/colab_core/manifest.json | head -60

## Next

- Open a few `outputs/eval/colab_core/traces/<task>/expert.json` vs `default.json` and read them.
- Hand-grade ~5 final answers as an expert; check the judge's `scores/**/*.judge.json` agree.
- Then run the full sweep (remove `--task-ids`) and, once `scripts/summarize_eval.py` exists, pivot the scores.